In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (TPpred-LE)

This notebook processes and standardizes the **TPpred-LE** dataset to produce a clean and consistent collection of toxic peptide sequences for downstream analysis and machine learning applications. The original data are distributed across **training**, **validation**, and **test** splits, each providing peptide sequences and associated labels in separate files.

- **Toxic effect / endpoint:** toxic
- **Source:** TPpred-LE
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Reads peptide sequences from FASTA files** and corresponding label files from each data split (train, validation, and test).
- **Merges sequences and labels** into a unified representation.
- **Selects toxic peptides only**, retaining entries with `TXP = 1` and mapping this field to the canonical `label` column.
- **Concatenates all splits** into a single dataset with a standardized schema:
  - `sequence`: peptide amino-acid sequence
  - `label`: toxicity label (binary)
- **Performs duplicate sequence quality control**:
  - removes redundant sequences with consistent labels,
  - flags sequences with conflicting labels as erroneous.
- **Generates dataset metadata** using the centralized raw-data description spreadsheet.
- **Exports the curated dataset and metadata** to the standardized output directory.

In [2]:
name_source = "TPpred-LE"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants.
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
def read_file(folder):
    df_seq = read_fasta_doc(f"{PATH_INPUT}/{name_source}/{folder}/seqs.fasta")
    df_label = pd.read_csv(f"{PATH_INPUT}/{name_source}/{folder}/labels.csv")

    df = pd.concat([df_seq, df_label], axis=1)
    return df

In [4]:
df_test = read_file("test")

In [5]:
df_train = read_file("train")

In [6]:
df_val= read_file("val")

- Concatenate dataset

In [7]:
df_tppredle = pd.concat([df_test, df_train, df_val], ignore_index=True)

In [8]:
df_tppredle = (
    df_tppredle[df_tppredle["TXP"] == 1]
    .rename(columns={"TXP": "label"})
    [["sequence", "label"]])
df_tppredle.shape

(2345, 2)

- Checking duplicates

In [9]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_tppredle, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [10]:
df_full.shape

(2345, 2)

In [11]:
df_errors.shape

(0, 1)

- Working with metada

In [12]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [13]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_tppredle)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2023,
 'last update date': datetime.datetime(2023, 1, 31, 0, 0),
 'download date': Timestamp('2025-04-01 00:00:00'),
 'file format': 'csv',
 'peptide property': 'antimicrobial, toxic, antibacterial, antiinflammatory, anticancer, antifungal, cell-penetrating, cell-cell communication, antiparasitic, antihypertensive, quorum-sensing, drug delivery, antiangiogenic, polystyrene surface-binding',
 'dataset information': 'Positive',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'No information',
 'repository or server': 'http://bliulab.net/TPpred-LE/data/',
 'publication': 'https://bmcbiol.biomedcentral.com/articles/10.1186/s12915-023-01740-w#availability-of-data-and-materials',
 'number_of_raw_sequences': 2345,
 'number_of_sequences_retained': 2345,
 'number_of_positive_sequences': 2345,
 'number_of_negative_sequences': 0,
 'number_of_erroneous_sequences': 

- Exporting data

In [14]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [15]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)